In [1]:
# LIBRERIAS

import os
import gc
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mongodb import MongoDBAtlasVectorSearch
from huggingface_hub import hf_hub_download


In [2]:
# --- 1. CONFIGURACIÓN ---
MODEL_EMBEDDING = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
URI_MONGO = os.environ.get('cs_mongo')
DB_NAME = "rag"
COL_NAME = "documents"

In [3]:
# --- 2. DESCARGA DEL MODELO GGUF ---
# Usamos Qwen 2.5 3B en versión Q4_K_M
REPO_ID = "Qwen/Qwen2.5-3B-Instruct-GGUF"
FILENAME = "qwen2.5-3b-instruct-q4_k_m.gguf"

print(f"Descargando modelo optimizado ({FILENAME})...")
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    cache_dir="./modelos_gguf" # Lo guardamos en una carpeta local
)
print(f"Modelo descargado en: {model_path}")

Descargando modelo optimizado (qwen2.5-3b-instruct-q4_k_m.gguf)...
Modelo descargado en: ./modelos_gguf\models--Qwen--Qwen2.5-3B-Instruct-GGUF\snapshots\7dabda4d13d513e3e842b20f0d435c732f172cbe\qwen2.5-3b-instruct-q4_k_m.gguf


In [4]:
# --- 3. CARGA DEL MOTOR (Llama.cpp) ---

n_gpu_layers = 0  # 0 porque usamos CPU
n_batch = 512     # Tamaño del bloque a procesar

# Callback para dar efecto streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=n_gpu_layers,
    n_batch=n_batch,
    n_ctx=4096,           # Ventana de contexto (cuánto texto recuerda)
    f16_kv=True,          # Optimización de memoria
    callback_manager=callback_manager,
    verbose=False,        # Para que no llene la pantalla de logs técnicos
    temperature=0.1,      # Precisión
    max_tokens=512,       # Longitud de respuesta
    n_threads=8           # Forzamos a 8 núcleos para que vaya más rápido
)

In [5]:
# --- 4. CONEXIÓN A BASE DE DATOS ---
print("Conectando a la biblioteca...")
embeddings = HuggingFaceEmbeddings(model_name=MODEL_EMBEDDING)
vectorstore = MongoDBAtlasVectorSearch.from_connection_string(
    connection_string=URI_MONGO,
    namespace=f"{DB_NAME}.{COL_NAME}",
    embedding=embeddings,
    index_name="vector_index"
)

print("\n ¡SISTEMA LISTO!")
print("*"*60)

Conectando a la biblioteca...

 ¡SISTEMA LISTO!
************************************************************


In [6]:
# --- 5. BUCLE DE CHAT ---
def iniciar_chat():
    while True:
        try:
            query = input("\n👤 Tú: ")
            if query.lower() in ["salir", "exit", "adios","Adios","Salir","Exit","Quit"]: break
            if not query.strip(): continue

            print(f"  Buscando...", end="\r")
            docs = vectorstore.similarity_search(query, k=3)
            
            if not docs:
                print("\n No encontré info en los libros.")
                continue

            contexto = "\n".join([d.page_content for d in docs])
            
            # Prompt específico para Qwen
            prompt_template = f"""<|im_start|>system
Eres un experto en economía. Responde a la pregunta basándote SOLO en el contexto. Responde en español.<|im_end|>
<|im_start|>user
Contexto:
{contexto}

Pregunta: {query}<|im_start|>assistant
"""
            print(f"   Generando respuesta...\n")
            # Invocamos al modelo (el texto saldrá escribiéndose solo gracias al callback)
            llm.invoke(prompt_template)
            
            print("\n" + "-"*60)

        except Exception as e:
            print(f"Error: {e}")

iniciar_chat()


👤 Tú:  Explícame la diferencia entre el valor de uso y el valor de cambio según Adam Smith


   Generando respuesta...

Según Adam Smith en su obra "The Wealth of Nations", hay una distinción importante entre el valor de uso (use value) y el valor de cambio (exchange value).

1. Valor de Uso: Este es el valor que tiene un bien o servicio para la satisfacción de las necesidades del individuo.

2. Valor de Cambio: Este es el valor que tiene un bien o servicio en términos de su capacidad para intercambiar con otros bienes y servicios.

En resumen, Adam Smith enfatizó que mientras el valor de uso se refiere a la satisfacción personal, el valor de cambio se refiere a la capacidad del bien para ser utilizado como medio de intercambio.
------------------------------------------------------------



👤 Tú:  Adios
